TASK 3: Real-world data aquisition and handling

In [ ]:
from datetime import datetime

import neurokit2 as nk
import numpy as np
import matplotlib.pyplot as plt
import scipy
from matplotlib.ticker import MaxNLocator
from datetime import timedelta

# Import Pandas only for reading Watch CSV File and for plotting description in Q1
import pandas as pd


In [ ]:
def load_bp_csv(filename, start_datetime):
    test = pd.read_csv(filename, skiprows=8, header=1, delimiter='\t')
    test = test.drop(0)
    test = test.drop(test.columns[3], axis=1)  # Drop the first column
    test = test.rename(columns={'CH9': 'ecg', 'CH1': 'ppg'})
    test['ppg'] = pd.to_numeric(test['ppg'], errors='coerce')
    # if 'min' in test.columns, create new milli sec column
    if 'min' in test.columns:
        # Convert 'min' to milliseconds
        test['time_ms'] = test['min'] * 60 * 1000
    else:
        test = test.rename(columns={'milliSec': 'time_ms'})
    test['ecg'] = pd.to_numeric(test['ecg'], errors='coerce')
    test['timestamp'] = pd.to_datetime(start_datetime) + pd.to_timedelta(test['time_ms'], unit='ms')
    return test


def load_watch_ecg(filename):
    df = pd.read_csv(filename, skiprows=2)
    df = df[['Timestamp', 'ECG data']].dropna()
    df.rename(columns={'Timestamp': 'timestamp', 'ECG data': 'ecg'}, inplace=True)
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce')
    df['ecg'] = pd.to_numeric(df['ecg'], errors='coerce')
    df = df.dropna()

    # Convert to seconds
    df['time_ms'] = (df['timestamp'] - df['timestamp'].iloc[0])
    return df


def load_watch_data(ecg_file, ppg_file):
    ecg_df = pd.read_csv(ecg_file, skiprows=2)
    ppg_df = pd.read_csv(ppg_file, skiprows=2)

    # Rename columns for clarity
    ecg_df.rename(columns={'Timestamp': 'timestamp', 'ECG data': 'ecg'}, inplace=True)
    ppg_df.rename(columns={'PPG Timestamp': 'timestamp', 'PPG data': 'ppg'}, inplace=True)

    latest_start_time = max(ecg_df['timestamp'].iloc[0], ppg_df['timestamp'].iloc[0])
    earliest_end_time = min(ecg_df['timestamp'].iloc[-1], ppg_df['timestamp'].iloc[-1])

    # Filter data to the common time range
    ecg_df = ecg_df[(ecg_df['timestamp'] >= latest_start_time) & (ecg_df['timestamp'] <= earliest_end_time)]
    ppg_df = ppg_df[(ppg_df['timestamp'] >= latest_start_time) & (ppg_df['timestamp'] <= earliest_end_time)]

    # Drop all columns except timestamp, time_ms and ecg/ppg
    ppg_df = ppg_df.drop(columns=['ADXL Timestamp'])
    ecg_df = ecg_df.drop(columns=['Seq No.'])
    # Merge ECG and PPG based on the millisecond given
    merged_df = pd.merge_asof(ecg_df.sort_values('timestamp'), ppg_df.sort_values('timestamp'), on='timestamp',
                              direction='nearest')

    merged_df['timestamp'] = pd.to_datetime(merged_df['timestamp'], unit='ms')
    return merged_df


def plot_watch_bp_ecg(watch_df, bp_data, start_time, end_time, dataset_name='Recording', vlines=[], vlines_name=[]):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    watch_df.sort_values('timestamp', inplace=True)
    bp_data.sort_values('time_ms', inplace=True)

    # Select time frame to plot
    # Discard time frames that are not represented in both recordings
    latest_start_time = max(watch_df['timestamp'].iloc[0], bp_data['timestamp'].iloc[0])  # + timedelta(minutes=5)
    earliest_end_time = min(watch_df['timestamp'].iloc[-1], bp_data['timestamp'].iloc[-1])
    watch_df = watch_df[(watch_df['timestamp'] >= latest_start_time) & (watch_df['timestamp'] <= earliest_end_time)]
    watch_df = watch_df[(watch_df['timestamp'] >= start_time) & (watch_df['timestamp'] <= end_time)]
    bp_data = bp_data[(bp_data['timestamp'] >= latest_start_time) & (bp_data['timestamp'] <= earliest_end_time)]
    bp_data = bp_data[(bp_data['timestamp'] >= start_time) & (bp_data['timestamp'] <= end_time)]
    watch_df = watch_df.dropna()
    # Plot ECG from Watch
    ax1.plot(watch_df['timestamp'], watch_df['PPG'], label='PPG from Watch', color='blue')
    ax1.set_ylabel('ECG Amplitude')
    ax1.set_xlabel('Time')
    ax1.set_title(f'{dataset_name} - PPG Signal from Smartwatch')

    # Plot ECG from BioPac
    ax2.plot(bp_data['timestamp'], bp_data['ecg'], label='ECG from BioPac', color='orange')
    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('ECG Amplitude')
    ax2.set_title(f'{dataset_name} - ECG Signal from BioPac')
    ax1.legend()
    ax2.legend()
    for vline, vline_name in zip(vlines, vlines_name):
        ax1.axvline(x=vline, color='red', linestyle='--', label=vline_name)
        ax2.axvline(x=vline, color='red', linestyle='--', label=vline_name)
    #ax1.legend()
    #ax2.legend()
    plt.tight_layout()
    plt.show()
    bp_data['ppg']
    fig, ax3 = plt.subplots(figsize=(12, 4))
    ax3.plot(bp_data['timestamp'], bp_data['ppg'], label='PPG from BioPac', color='orange')
    ax3.set_xlabel('Time')
    ax3.set_ylabel('ECG Amplitude')
    ax3.set_title(f'{dataset_name} - PPG Signal from BioPac')
    ax3.legend()
    for vline, vline_name in zip(vlines, vlines_name):
        ax3.axvline(x=vline, color='red', linestyle='--', label=vline_name)
    #ax3.legend()
    ax3.yaxis.set_major_locator(MaxNLocator(nbins=5))
    plt.tight_layout()
    plt.show()
    # plot xyz in one plot
    fig, ax4 = plt.subplots(figsize=(12, 4))
    ax4.plot(watch_df['timestamp'], watch_df['X'], label='X-axis', color='red')
    ax4.plot(watch_df['timestamp'], watch_df['Y'], label='Y-axis', color='green')
    ax4.plot(watch_df['timestamp'], watch_df['Z'], label='Z-axis', color='purple')
    ax4.set_xlabel('Time')
    ax4.set_ylabel('Acceleration')
    ax4.set_title(f'{dataset_name} - Acceleration Data from Watch')
    ax4.legend()
    for vline, vline_name in zip(vlines, vlines_name):
        ax4.axvline(x=vline, color='red', linestyle='--', label=vline_name)
    #ax4.legend()

    plt.tight_layout()
    plt.show()


def crop_window(data, start_time, end_time):
    data = data[(data['timestamp'] >= start_time) & (data['timestamp'] <= end_time)]
    return data

In [ ]:
# Synchonization is done in here by adjusting the time difference from the logged time and the actual time from smartwatch
watch_rest = load_watch_data('data/assignment_3/Rest/ecg_2025_06_12_12_57_44ecgcentrifuge.csv',
                             'data/assignment_3/Rest/ppg_2025_06_12_12_57_44ppgcentrifuge.csv')
#2025-06-12 12:42:40.750
bp_rest = load_bp_csv('data/assignment_3/Rest/Biopac.txt',
                      start_datetime=datetime(2025, 6, 12, 10, 58, 20, 193000) - timedelta(seconds=2.9))

watch_stairs = load_watch_data('data/assignment_3/Stairs/ecg_2025_06_12_12_00_41ecgcentrifuge.csv',
                               'data/assignment_3/Stairs/ppg_2025_06_12_12_00_41ppgcentrifuge.csv')
#2025-06-12 12:01:19.122
bp_stairs = load_bp_csv('data/assignment_3/Stairs/Biopac.txt',
                        start_datetime=datetime(2025, 6, 12, 10, 1, 3, 334000) - timedelta(seconds=-1.65))

watch_workout = load_watch_data('data/assignment_3/Workout/ecg_2025_06_12_12_42_06ecgcentrifuge.csv',
                                'data/assignment_3/Workout/ppg_2025_06_12_12_42_06ppgcentrifuge.csv')
#2025-06-12 12:42:40.750
bp_workout = load_bp_csv('data/assignment_3/Workout/Biopac.txt',
                         start_datetime=datetime(2025, 6, 12, 10, 42, 40, 750000) - timedelta(seconds=.4))




In [ ]:
# Synchronization
plot_watch_bp_ecg(watch_rest, bp_rest,
                  start_time=datetime(2025, 6, 12, 10, 58, 20, 193000),
                  end_time=datetime(2025, 6, 12, 11, 0, 0, 750000),
                  dataset_name='Workout')

In [ ]:
# Q1
cropped_stairs = crop_window(watch_stairs, start_time=datetime(2025, 6, 12, 10, 5, 0, 0),
                             end_time=datetime(2025, 6, 12, 10, 5, 10, 0))
print('Initial Length of DF for 10 seconds of Stairs: ', len(cropped_stairs))


# Resample to 1khz
def resample_watch_data(watch_df):
    watch_df = watch_df.set_index('timestamp')
    watch_df = watch_df.resample('1L').mean().interpolate(method='linear')
    watch_df = watch_df.reset_index()
    return watch_df


watch_stairs_resampled = resample_watch_data(watch_stairs)
cropped_stairs_2 = crop_window(watch_stairs_resampled, start_time=datetime(2025, 6, 12, 10, 5, 0, 0),
                               end_time=datetime(2025, 6, 12, 10, 5, 10, 0))

print('Length of DF after resampling and cropping: ', len(cropped_stairs_2))

# Plot the resampled data
#!!! There is a couple of seconds before and after exercise, to show the difference
plot_watch_bp_ecg(watch_stairs_resampled, bp_stairs,
                  start_time=datetime(2025, 6, 12, 10, 7, 0, 0),
                  end_time=datetime(2025, 6, 12, 10, 17, 45, 0),
                  dataset_name='Stairs',
                  vlines=[datetime(2025, 6, 12, 10, 7, 16, 0),
                          datetime(2025, 6, 12, 10, 17, 34, 0)],
                  vlines_name=['Start of Workout', 'End of Workout'])

In [ ]:
# Q2
cropped_workout = crop_window(watch_workout, start_time=datetime(2025, 6, 12, 10, 43, 40, 0),
                              end_time=datetime(2025, 6, 12, 10, 43, 41, 0))
print('Initial Length of DF for 10 seconds of Workout: ', len(cropped_workout))
watch_workout_resampled = resample_watch_data(watch_workout)
cropped_workout_2 = crop_window(watch_workout_resampled, start_time=datetime(2025, 6, 12, 10, 43, 40, 0),
                                end_time=datetime(2025, 6, 12, 10, 43, 41, 0))
print('Length of DF after resampling and cropping: ', len(cropped_workout_2))

# Plot the resampled data
plot_watch_bp_ecg(watch_workout, bp_workout,
                  start_time=datetime(2025, 6, 12, 10, 42, 47, 0),
                  end_time=datetime(2025, 6, 12, 11, 43, 00, 0),
                  dataset_name='Workout',
                  vlines=[
                      datetime(2025, 6, 12, 10, 42, 55, 0),
                      datetime(2025, 6, 12, 10, 43, 42, 0),

                      datetime(2025, 6, 12, 10, 44, 39, 0),
                      datetime(2025, 6, 12, 10, 45, 19, 0),

                      datetime(2025, 6, 12, 10, 46, 21, 0),
                      datetime(2025, 6, 12, 10, 46, 57, 0),

                      datetime(2025, 6, 12, 10, 47, 55, 0),
                      datetime(2025, 6, 12, 10, 48, 48, 0),

                      datetime(2025, 6, 12, 10, 50, 00, 0),
                      datetime(2025, 6, 12, 10, 50, 37, 0)],
                  vlines_name=['Transition'] * 10
                  )


In [ ]:
# Q3

def compute_heart_rate_metrics(ecg_df, sampling_rate=1000):
    # Schritt 1: R-Peak-Detection mit neurokit2
    ecg_signal = ecg_df['ecg'].fillna(method='ffill').fillna(method='bfill')
    ecg_cleaned = nk.ecg_clean(ecg_signal, sampling_rate=sampling_rate)
    _, rpeaks = nk.ecg_peaks(ecg_cleaned, sampling_rate=sampling_rate)
    rpeaks_indices = rpeaks['ECG_R_Peaks']

    # Schritt 2: Instantane Herzfrequenz (ohne Toolbox)
    rr_intervals = np.diff(ecg_df['timestamp'].values[rpeaks_indices]) / np.timedelta64(1, 's')
    inst_hr = 60 / rr_intervals
    inst_hr_time = ecg_df['timestamp'].values[rpeaks_indices][1:]

    inst_hr_df = pd.DataFrame({'timestamp': inst_hr_time, 'hr': inst_hr})

    # Schritt 3: Geglättete HR (Median über 10s mit 1s Schrittweite)
    window_size = 10
    step_size = 1
    start_time = inst_hr_df['timestamp'].min()
    end_time = inst_hr_df['timestamp'].max()

    smoothed_hr = []
    times = []

    current = start_time
    while current + timedelta(seconds=window_size) < end_time:
        window_end = current + timedelta(seconds=window_size)
        middle = current + timedelta(seconds=window_size / 2)

        hr_vals = inst_hr_df[(inst_hr_df['timestamp'] >= current) &
                             (inst_hr_df['timestamp'] < window_end)]['hr']
        if len(hr_vals) > 0:
            smoothed_hr.append(np.median(hr_vals))
            times.append(middle)
        current += timedelta(seconds=step_size)

    smoothed_hr_df = pd.DataFrame({'timestamp': times, 'smoothed_hr': smoothed_hr})

    return inst_hr_df, smoothed_hr_df, ecg_df, rpeaks_indices


def plot_ecg_with_rpeaks(ecg_df, rpeaks_indices, title, start_time, end_time):
    segment = ecg_df[(ecg_df['timestamp'] >= start_time) & (ecg_df['timestamp'] <= end_time)].copy()
    segment_indices = segment.index

    rpeaks_in_segment = [i for i in rpeaks_indices if i in segment_indices]

    plt.figure(figsize=(12, 4))
    plt.plot(segment['timestamp'], segment['ecg'], label='ECG')
    plt.plot(ecg_df['timestamp'].iloc[rpeaks_in_segment],
             ecg_df['ecg'].iloc[rpeaks_in_segment], 'ro', label='R-peaks')
    plt.title(title)
    plt.xlabel('Time')
    plt.ylabel('Amplitude')
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_hr(inst_hr_df, smoothed_hr_df, title):
    plt.figure(figsize=(12, 4))
    plt.plot(inst_hr_df['timestamp'], inst_hr_df['hr'], label='Instant HR', alpha=0.7)
    plt.plot(smoothed_hr_df['timestamp'], smoothed_hr_df['smoothed_hr'], label='Smoothed HR', linewidth=2)
    plt.title(title)
    plt.ylabel('Heart Rate (bpm)')
    plt.xlabel('Time')
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# Q3 
# Schritt 1: Auf eine Biopac-ECG-Messung anwenden (z. B. "Stairs")
inst_hr_df, smoothed_hr_df, ecg_clean_df, rpeaks = compute_heart_rate_metrics(bp_stairs)

# Schritt 2: 20s Segment – wo es gut funktioniert hat (z. B. manuell wählen)
plot_ecg_with_rpeaks(ecg_clean_df, rpeaks,
                     title="Good R-Peak Detection",
                     start_time=datetime(2025, 6, 12, 10, 2, 0),
                     end_time=datetime(2025, 6, 12, 10, 2, 20))

# Schritt 3: 20s Segment – wo es schlecht funktioniert hat (manuell wählen)
plot_ecg_with_rpeaks(ecg_clean_df, rpeaks,
                     title="Faulty R-Peak Detection",
                     start_time=datetime(2025, 6, 12, 10, 7, 0),
                     end_time=datetime(2025, 6, 12, 10, 7, 20))

# Schritt 4: Vergleich der Herzfrequenz-Verläufe
plot_hr(inst_hr_df, smoothed_hr_df, title="Heart Rate: Instant vs. Smoothed (Stairs)")


In [ ]:
def compute_heart_rate_metrics_mean(ecg_df, sampling_rate=1000):
    import neurokit2 as nk

    # Schritt 1: R-Peak-Erkennung mit neurokit2
    ecg_signal = ecg_df['ecg'].fillna(method='ffill').fillna(method='bfill')
    ecg_cleaned = nk.ecg_clean(ecg_signal, sampling_rate=sampling_rate)
    _, rpeaks = nk.ecg_peaks(ecg_cleaned, sampling_rate=sampling_rate)
    rpeaks_indices = rpeaks['ECG_R_Peaks']

    # Schritt 2: Instantane Herzfrequenz berechnen (60 / RR-Intervall in s)
    rr_intervals = np.diff(ecg_df['timestamp'].values[rpeaks_indices]) / np.timedelta64(1, 's')
    inst_hr = 60 / rr_intervals
    inst_hr_time = ecg_df['timestamp'].values[rpeaks_indices][1:]

    inst_hr_df = pd.DataFrame({'timestamp': inst_hr_time, 'hr': inst_hr})

    # Schritt 3: Smoothed HR (Mittelwert über 10s, 1s Schritt)
    window_size = 10
    step_size = 1
    start_time = inst_hr_df['timestamp'].min()
    end_time = inst_hr_df['timestamp'].max()

    smoothed_hr = []
    times = []

    current = start_time
    while current + timedelta(seconds=window_size) < end_time:
        window_end = current + timedelta(seconds=window_size)
        middle = current + timedelta(seconds=window_size / 2)

        hr_vals = inst_hr_df[(inst_hr_df['timestamp'] >= current) &
                             (inst_hr_df['timestamp'] < window_end)]['hr']
        if len(hr_vals) > 0:
            smoothed_hr.append(np.mean(hr_vals))
            times.append(middle)
        current += timedelta(seconds=step_size)

    smoothed_hr_df = pd.DataFrame({'timestamp': times, 'smoothed_hr': smoothed_hr})

    return inst_hr_df, smoothed_hr_df, ecg_df, rpeaks_indices


In [ ]:
# Q4
# Berechne Herzfrequenzen
inst_hr_df, smoothed_hr_df, ecg_df, rpeaks = compute_heart_rate_metrics_mean(bp_workout)

# Plot: gutes Segment (z. B. 10:43:20–10:43:40)
plot_ecg_with_rpeaks(ecg_df, rpeaks,
                     title="Good R-Peak Detection (Workout)",
                     start_time=datetime(2025, 6, 12, 10, 43, 20),
                     end_time=datetime(2025, 6, 12, 10, 43, 40))

# Plot: schlechtes Segment (z. B. 10:44:30–10:44:50)
plot_ecg_with_rpeaks(ecg_df, rpeaks,
                     title="Faulty R-Peak Detection (Workout)",
                     start_time=datetime(2025, 6, 12, 10, 44, 30),
                     end_time=datetime(2025, 6, 12, 10, 44, 50))

# Plot: Herzfrequenzverlauf
plot_hr(inst_hr_df, smoothed_hr_df, title="Heart Rate – Instant vs Smoothed (Workout)")



In [ ]:
# Q5
def compute_ppg_hr_metrics(ppg_df, sampling_rate=1000):
    import neurokit2 as nk

    ppg_signal = ppg_df['PPG'].fillna(method='ffill').fillna(method='bfill')
    ppg_clean = nk.ppg_clean(ppg_signal, sampling_rate=sampling_rate)
    _, peaks = nk.ppg_peaks(ppg_clean, sampling_rate=sampling_rate)
    peak_indices = peaks['PPG_Peaks']

    # Instantane Herzfrequenz
    timestamps = ppg_df['timestamp'].values
    rr_intervals = np.diff(timestamps[peak_indices]) / np.timedelta64(1, 's')
    hr = 60 / rr_intervals
    hr_time = timestamps[peak_indices][1:]

    inst_hr_df = pd.DataFrame({'timestamp': hr_time, 'hr': hr})

    # Smoothed HR: Mean in 10s window, 1s slide
    window_size = 10
    step_size = 1
    start_time = inst_hr_df['timestamp'].min() + timedelta(seconds=5)
    end_time = inst_hr_df['timestamp'].max()

    smoothed_hr, times = [], []
    current = start_time
    while current + timedelta(seconds=window_size) < end_time:
        window_end = current + timedelta(seconds=window_size)
        middle = current + timedelta(seconds=window_size / 2)
        hr_vals = inst_hr_df[(inst_hr_df['timestamp'] >= current) &
                             (inst_hr_df['timestamp'] < window_end)]['hr']
        if len(hr_vals) > 0:
            smoothed_hr.append(np.mean(hr_vals))
            times.append(middle)
        current += timedelta(seconds=step_size)

    smoothed_df = pd.DataFrame({'timestamp': times, 'smoothed_hr': smoothed_hr})
    return inst_hr_df, smoothed_df, ppg_df, peak_indices


def plot_ppg_with_peaks(ppg_df, peak_indices, title, start_time, end_time):
    segment = ppg_df[(ppg_df['timestamp'] >= start_time) & (ppg_df['timestamp'] <= end_time)]
    segment_indices = segment.index
    peaks_in_segment = [i for i in peak_indices if i in segment_indices]

    plt.figure(figsize=(12, 4))
    plt.plot(segment['timestamp'], segment['PPG'], label='PPG')
    plt.plot(ppg_df['timestamp'].iloc[peaks_in_segment],
             ppg_df['PPG'].iloc[peaks_in_segment], 'ro', label='Peaks')
    plt.title(title)
    plt.xlabel('Time')
    plt.ylabel('PPG Amplitude')
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_ppg_hr(inst_hr_df, smoothed_hr_df, title):
    plt.figure(figsize=(12, 4))
    plt.plot(inst_hr_df['timestamp'], inst_hr_df['hr'], label='Instant HR', alpha=0.6)
    plt.plot(smoothed_hr_df['timestamp'], smoothed_hr_df['smoothed_hr'],
             label='Smoothed HR (10s mean)', linewidth=2)
    plt.title(title)
    plt.ylabel("Heart Rate (bpm)")
    plt.xlabel("Time")
    plt.legend()
    plt.tight_layout()
    plt.show()



In [ ]:
#Q5

# Herzfrequenz aus Smartwatch-PPG ("Stairs")
inst_hr_ppg, smoothed_hr_ppg, ppg_df, peak_idx = compute_ppg_hr_metrics(watch_stairs)

# Plot: gutes Segment
plot_ppg_with_peaks(ppg_df, peak_idx,
                    title="Good PPG Peak Detection (Stairs)",
                    start_time=datetime(2025, 6, 12, 10, 2, 0),
                    end_time=datetime(2025, 6, 12, 10, 2, 20))

# Plot: schlechtes Segment
plot_ppg_with_peaks(ppg_df, peak_idx,
                    title="Faulty PPG Peak Detection (Stairs)",
                    start_time=datetime(2025, 6, 12, 10, 4, 0),
                    end_time=datetime(2025, 6, 12, 10, 4, 20))

# Plot: HR-Verlauf
plot_ppg_hr(inst_hr_ppg, smoothed_hr_ppg, title="Smartwatch PPG HR – Instant vs Smoothed (Stairs)")


In [ ]:
watch_stairs

In [ ]:
# Q6
# Herzfrequenz aus Smartwatch-PPG ("Stairs")
bp_stairs.rename(columns={'ppg': 'PPG'}, inplace=True)
inst_hr_ppg, smoothed_hr_ppg, ppg_df, peak_idx = compute_ppg_hr_metrics(bp_stairs)

# Plot: gutes Segment
plot_ppg_with_peaks(ppg_df, peak_idx,
                    title="Good PPG Peak Detection (Stairs)",
                    start_time=datetime(2025, 6, 12, 10, 2, 0),
                    end_time=datetime(2025, 6, 12, 10, 2, 20))

# Plot: schlechtes Segment
plot_ppg_with_peaks(ppg_df, peak_idx,
                    title="Faulty PPG Peak Detection (Stairs)",
                    start_time=datetime(2025, 6, 12, 10, 4, 0),
                    end_time=datetime(2025, 6, 12, 10, 4, 20))

# Plot: HR-Verlauf
plot_ppg_hr(inst_hr_ppg, smoothed_hr_ppg, title="Smartwatch PPG HR – Instant vs Smoothed (Stairs)")


In [ ]:
# Q7



In [ ]:
# Q8



In [ ]:
# Q9

